# Phase III — GitHub-powered one-run demonstration
Run the cell below after publishing the two files in the public repository `torobbycordova-81/Phase-III-Demo` under `data/`. The notebook downloads and verifies both files, predicts, and shows three metrics plus one figure. No upload dialogs.

**Validation:** 196 observations from DOI 10.1016/j.marstruc.2020.102881, held out in grouped-CV fold 4. This is *not* independent external validation. The CSV is already harmonized; this notebook performs model-specific transforms and input checks, not raw-publication ingestion.


In [ ]:
# ONE RUN — download the trusted model and held-out dataset from a public GitHub repository.
# Change REPO only if you chose a different repository name.
from pathlib import Path
import hashlib, urllib.request, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

REPO = "torobbycordova-81/Phase-III-Demo"
BRANCH = "main"
BASE = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data"
FILES = {
    "phase3_grouped_demo_fold_model.joblib": "8478089d4a568d0a4abadfc8139de6a969b748a75312992b8e69f8dbe0d6e724",
    "phase3_demo_publication_inputs.csv": "bf60cc4af98185aa2c4ff8d64ea541e992ab0b0adb0613559194a490a2c4d25a",
}
for filename, expected_sha256 in FILES.items():
    url = f"{BASE}/{filename}"
    try:
        with urllib.request.urlopen(url, timeout=45) as response:
            contents = response.read()
    except Exception as exc:
        raise RuntimeError(f"Could not download {filename} from {url}. Confirm repository is public and files were uploaded to data/ on main.") from exc
    if hashlib.sha256(contents).hexdigest() != expected_sha256:
        raise ValueError(f"SHA-256 mismatch for {filename}: GitHub file differs from the verified demo package.")
    Path(filename).write_bytes(contents)

# This .joblib file is a trusted project artifact verified above. Never load unknown joblib files.
df = pd.read_csv("phase3_demo_publication_inputs.csv")
required = ['deltaK_MPa_sqrt_m','R_ratio','temperature_C','frequency_Hz',
            'environment_canonical','material_family_canonical','manufacturing_method','measured_dadN']
missing = [c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing required columns: {missing}")
for col in ['deltaK_MPa_sqrt_m','R_ratio','temperature_C','frequency_Hz','measured_dadN']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
if df.empty or df[['deltaK_MPa_sqrt_m','measured_dadN']].isna().any().any():
    raise ValueError('Missing or invalid ΔK or measured crack growth rate.')
if (df.deltaK_MPa_sqrt_m <= 0).any() or (df.measured_dadN <= 0).any():
    raise ValueError('ΔK and measured crack growth rate must be positive.')
if (df.frequency_Hz.dropna() < 0).any():
    raise ValueError('Frequency cannot be negative.')

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='.*Trying to unpickle estimator.*')
    model = joblib.load('phase3_grouped_demo_fold_model.joblib')
X = df[['R_ratio','temperature_C','frequency_Hz',
        'environment_canonical','material_family_canonical','manufacturing_method']].copy()
X['ln_deltaK'] = np.log(df['deltaK_MPa_sqrt_m'])
X['frequency_Hz'] = np.log1p(X['frequency_Hz'])
pred_ln = model.predict(X)
measured_ln = np.log(df['measured_dadN'].to_numpy())
if not np.isfinite(pred_ln).all(): raise ValueError('Non-finite prediction.')
mae = mean_absolute_error(measured_ln, pred_ln)
rmse = np.sqrt(mean_squared_error(measured_ln, pred_ln))
r2 = r2_score(measured_ln, pred_ln)
result = df.copy()
result['predicted_dadN'] = np.exp(pred_ln)
result['predicted_ln_dadN'] = pred_ln
result.to_csv('phase3_demo_predictions_generated.csv', index=False)

print('\nPUBLICATION-HELD-OUT MODEL DEMONSTRATION')
print(f'Observations: {len(df)}  |  R² (ln scale): {r2:.3f}  |  MAE (ln scale): {mae:.3f}  |  RMSE (ln scale): {rmse:.3f}')
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['deltaK_MPa_sqrt_m'], df['measured_dadN'], s=25, alpha=.72, label='Measured')
ax.scatter(df['deltaK_MPa_sqrt_m'], result['predicted_dadN'], s=25, alpha=.72, marker='x', label='Predicted')
ax.set_yscale('log')
ax.set_xlabel('Stress intensity factor range, ΔK (MPa√m)')
ax.set_ylabel('Crack growth rate, da/dN (training target units/cycle)')
ax.set_title('Publication-held-out steel: measured vs. predicted')
ax.grid(alpha=.20)
ax.legend()
fig.tight_layout()
fig.savefig('phase3_demo_result.png', dpi=180)
plt.show()
print('Predictions saved to phase3_demo_predictions_generated.csv')
print('Note: publication-held-out cross-validation, not independent external validation.')
